# Project Scaffolding with uv — A New Hope for Bob

*Bob scaffolds a backend and frontend in one uv workspace, then follows the projects through packaging.*

Follow [Chapter 03, Section 01](../../docs/chapter-03/section-01.md) for the reference explanation. Run these cells in order in this lab's Dev Container. All generated projects live in `lab/`; the Jupyter kernel stays on the container's system Python.

## Introduction — One Tool, Many Jobs

> **Bob:** I used to keep a checklist for Python, venv, pip, and packaging. Alice says uv can handle the whole trip. A new hope? I'll believe it when I see the files.

On macOS or Linux, install uv with `curl -LsSf https://astral.sh/uv/install.sh | sh`. On Windows, use PowerShell: `powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"`. See [Section 01](../../docs/chapter-03/section-01.md#install-uv) for the copyable commands. The Dev Container has uv installed already; check its version:

In [ ]:
%%bash
uv --version

## Manage Python with uv

> **Bob:** By Jar Jar's eyes! (Binks, editor's note.) I thought installing an interpreter was a job for the system package manager. Now you're telling me uv can download one itself, without replacing the system Python? The Force is strong with this tool.

Like `nvm` for Node.js and `rustup` for Rust, `uv` can install and select language versions. Use `uv python` to manage interpreters separately from the system installation.

### Installing Python with uv

Install Python 3.12 for both workspace components; the notebook kernel stays on the container's system Python:

In [ ]:
%%bash
uv python install 3.12

## Set Up a uv Workspace

### Create the Workspace

> **Bob:** Two projects in one repository, but one lockfile? That could trim a few pages from my checklist.

A uv workspace lets related projects keep their own `pyproject.toml` files while sharing dependency resolution and a single `uv.lock`. Reset only the generated `lab/` directory and create it for this exercise:

In [ ]:
from pathlib import Path
import shutil

lab = Path('lab')
if lab.exists():
    shutil.rmtree(lab)
lab.mkdir()
print('Clean lab directory:', lab.resolve())

Initialize a bare root project with Python 3.12. It will own the workspace configuration and, after dependency resolution, the shared lockfile:

In [ ]:
%%bash
uv init --bare --python 3.12 --vcs none lab/uv-workspace

Create the member directories before pinning their interpreters:

In [ ]:
(Path('lab/uv-workspace/backend')).mkdir()
(Path('lab/uv-workspace/frontend')).mkdir()
print('Created backend and frontend member directories')

### Pin Python for the Components

Both components use Python 3.12. `.python-version` selects the local interpreter, while `requires-python` in each `pyproject.toml` declares supported versions. Workspace members must have compatible requirements.

> **Bob:** Same Python pin for both sides of the app. At least this part of the flight plan agrees.

#### Pin the Backend

Pin the backend before its project file exists. `--no-project` skips compatibility checks against a project:

In [ ]:
%%bash
uv --directory lab/uv-workspace/backend python pin --no-project 3.12

#### Pin the Frontend

Pin the frontend to the same interpreter version:

In [ ]:
%%bash
uv --directory lab/uv-workspace/frontend python pin --no-project 3.12

Verify that both local pins match:

In [ ]:
for component in ('backend', 'frontend'):
    pin = Path('lab/uv-workspace', component, '.python-version')
    print(f'{component}: {pin.read_text().strip()}')

### Initialize the Workspace Components

Each component will be a packaged application with its own project metadata and generated starter package. The workspace root will list both members.

#### Initialize the Backend

> **Bob:** A Bottle backend and a Streamlit frontend. Finally, something lightweight at both ends.

Initialize the backend as a packaged application. Disabling VCS initialization leaves Git ownership with the workspace root:

In [ ]:
%%bash
uv init --app --package --vcs none --author-from none lab/uv-workspace/backend

Inspect the generated backend layout and project metadata. The starter module is scaffolding; the lab does not implement a Bottle route:

In [ ]:
backend = Path('lab/uv-workspace/backend')
print('Files:', sorted(str(p.relative_to(backend)) for p in backend.rglob('*') if p.is_file()))
print(backend.joinpath('pyproject.toml').read_text())

#### Initialize the Frontend

Initialize the frontend as another packaged application in the workspace. Streamlit will be its runtime dependency, added after inspecting this generated scaffold:

In [ ]:
%%bash
uv init --app --package --vcs none --author-from none lab/uv-workspace/frontend

Inspect the generated frontend files and complete project configuration. No Streamlit UI module is implemented in this scaffolding section:

In [ ]:
frontend = Path('lab/uv-workspace/frontend')
print('Files:', sorted(str(p.relative_to(frontend)) for p in frontend.rglob('*') if p.is_file()))
print(frontend.joinpath('pyproject.toml').read_text())

Inspect the root workspace configuration. It should list the backend and frontend members; the root lockfile will be created when dependencies are first resolved:

In [ ]:
root = Path('lab/uv-workspace')
print('Root pyproject.toml:\n', root.joinpath('pyproject.toml').read_text())

### Add a Workspace Dependency

The backend and frontend already share a workspace. Bob will add a small generated library as another member, then make the backend depend on it through the workspace.

> **Bob:** The shared lock is already doing the coordination. If the ships need to share a library, the workspace can keep it local.

Create a directory for the library and initialize it as a workspace member. The library's generated source remains untouched:

In [ ]:
%%bash
mkdir -p lab/uv-workspace/packages
uv init --lib --vcs none --author-from none lab/uv-workspace/packages/shared-components

Inspect the updated root member list and the generated library metadata:

In [ ]:
print(Path('lab/uv-workspace/pyproject.toml').read_text())
print(Path('lab/uv-workspace/packages/shared-components/pyproject.toml').read_text())

Add the workspace library to the backend project. The workspace source tells uv to use the local member rather than looking for a registry package:

In [ ]:
%%bash
uv --directory lab/uv-workspace/backend add --workspace shared-components

The backend metadata now declares the local workspace dependency, and the shared lock remains at the root:

In [ ]:
print(Path('lab/uv-workspace/backend/pyproject.toml').read_text())
print('Root lockfile:', Path('lab/uv-workspace/uv.lock').exists())
print('Library member lockfile:', Path('lab/uv-workspace/packages/shared-components/uv.lock').exists())

## Work in the Project Environment

### Add Dependencies and Run the Project

> **Info:** By default, `uv add` updates project metadata and the workspace `uv.lock`, creates `.venv` if needed, and installs dependencies and the packaged project. Workspace members share dependency resolution and use one root lockfile. See [Dependency Management with uv](../../docs/chapter-03/section-02.md#synchronizing) for synchronization workflows.

> **Bob:** No activation script, and dependencies get written into the right component? Let's equip both ships.

Add Bottle to the backend as an exact runtime requirement:

In [ ]:
%%bash
uv --directory lab/uv-workspace/backend add "bottle==0.13.4"

Add Streamlit to the frontend runtime dependencies:

In [ ]:
%%bash
uv --directory lab/uv-workspace/frontend add streamlit

Inspect the backend configuration after Bottle was added:

In [ ]:
print(Path('lab/uv-workspace/backend/pyproject.toml').read_text())

Inspect the frontend configuration after Streamlit was added:

In [ ]:
print(Path('lab/uv-workspace/frontend/pyproject.toml').read_text())

Both dependency additions resolve through the workspace. Verify the one root lockfile and confirm the member directories do not own separate lockfiles:

In [ ]:
print('Root lockfile:', Path('lab/uv-workspace/uv.lock').exists())
for component in ('backend', 'frontend'):
    print(f'{component} lockfile:', Path('lab/uv-workspace', component, 'uv.lock').exists())

Render the generated workspace tree. This makes the two component pins and the single root lockfile easy to distinguish:

In [ ]:
def show_tree(directory: Path, prefix: str = '') -> None:
    children = sorted(
        (path for path in directory.iterdir() if path.name not in {'.venv', 'dist'}),
        key=lambda path: (path.is_file(), path.name),
    )
    for index, path in enumerate(children):
        last = index == len(children) - 1
        print(prefix + ('└── ' if last else '├── ') + path.name + ('/' if path.is_dir() else ''))
        if path.is_dir():
            show_tree(path, prefix + ('    ' if last else '│   '))

workspace = Path('lab/uv-workspace')
print(workspace.name + '/')
show_tree(workspace)

Add Ruff and pytest to the backend's `dev` dependency group, not runtime requirements.

In [ ]:
%%bash
uv --directory lab/uv-workspace/backend add --dev ruff pytest

> **Version Note:** Your Ruff and pytest versions may differ. `uv.lock` records the exact resolved versions.

### Use the `[tool.*]` Table

Tools that support `pyproject.toml` can keep settings alongside project metadata. Add Ruff's line length and pytest's test directory and reporting options to the backend project. These tables configure tools; they do not install them:

In [ ]:
import tomllib

backend_project = Path('lab/uv-workspace/backend/pyproject.toml')
project_text = backend_project.read_text()
config = tomllib.loads(project_text)
if 'ruff' not in config.get('tool', {}):
    project_text += '\n[tool.ruff]\nline-length = 88\n'
if 'pytest' not in config.get('tool', {}):
    project_text += '\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\naddopts = "-ra"\n'
tomllib.loads(project_text)
backend_project.write_text(project_text)
print(backend_project.read_text())

The backend and frontend each have their own project metadata and dependencies. The workspace root owns the shared lockfile; the generated application modules remain untouched.

## Build a Package

### Understand the Build Backend

In packaging terminology, a build frontend coordinates a build and invokes the configured build backend. That backend creates installable distributions. This terminology is separate from the Streamlit frontend component. The packaged application templates use `uv_build` in `[build-system]`; the `uv build` command invokes the configured backend.

### Create Distributions

> **Bob:** A wheel and a source archive from one command? That's a much smaller release checklist.

Build the backend distributions:

In [ ]:
%%bash
uv build lab/uv-workspace/backend --out-dir lab/uv-workspace/backend/dist

Build the frontend distributions:

In [ ]:
%%bash
uv build lab/uv-workspace/frontend --out-dir lab/uv-workspace/frontend/dist

Inspect the generated backend distributions. The `.whl` is installable; the `.tar.gz` is a source distribution:

In [ ]:
print('Backend distributions:', sorted(p.name for p in Path('lab/uv-workspace/backend/dist').iterdir()))

## Publish a Package

### Upload to a Package Index

`uv publish` uploads a component's distributions to PyPI; it needs valid package metadata and authentication, such as a PyPI token in `UV_PUBLISH_TOKEN`. For a practice release, use a TestPyPI token and `--publish-url https://test.pypi.org/legacy/`. This lab only explains the commands and does not upload either component.

> **Bob:** Publishing is where I'll check the manifest twice. No need to send my training droid to the real galaxy today.

## Epilogue — Bob's New Checklist

> **Bob:** I started with one command, and now I know where both components keep their settings, how they share a lockfile, and which Python they use. Alice might have been right.

| Bob's task | uv command or file |
| --- | --- |
| Start a workspace | Root `pyproject.toml` and `[tool.uv.workspace]` |
| Pin component Python | `uv python pin`, component `.python-version` files |
| Initialize backend and frontend | `uv init --app --package` |
| Add Bottle and Streamlit | `uv add`, component `pyproject.toml` files, root `uv.lock` |
| Configure, build, and publish | `[tool.*]`, `uv_build`, `uv build`, then `uv publish` when ready |

Next, [Bob discovers the Force of uv](../uv_dependency_lab/uv_dependency_lab.ipynb): declaring, locking, and syncing dependencies in depth.